# Clustering audio à partir des fichiers chroma

On utilise ici les features de Chordino qui permettent d'étudier les accords.

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import seaborn as sns
import plotly.express as px
from plotly.offline import plot
import plotly.io as pio
from sklearn.manifold import MDS
import librosa
from scipy.cluster.hierarchy import linkage, dendrogram

In [12]:
dico_res = {"maj":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,0],
"min":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,0,0],
"dim_dim7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1,0,0],
"dim_min7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,1,0],
"maj_min7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,1,0],
"maj_maj7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,1,0,0,0,1],
"min_min7":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,0,1,0],
"dim":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0],
"aug":[0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0]}

In [ ]:
dico_notes = {
    "A" : 0,
    "A#" : 1,
    "Bb" : 1,
    "B" : 2,
    "C" : 3,
    "C#" : 4,
    "Db" : 4,
    "D" : 5,
    "D#" : 6,
    "Eb" : 6,
    "E" : 7,
    "F" : 8,
    "F#" : 9,
    "Gb" : 9,
    "G" : 10,
    "G#" : 11,
    "Ab" : 11,
    "N" : -1
}

In [11]:
chain = "Eb_maj_min7"
li = chain.split("_")
print(li[0],"_".join(li[1:]))

Eb maj_min7


In [13]:
["a"]*10

['a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a', 'a']

## Récupération d'un fichier et transformation en features

In [17]:
#A changer
def get_matrix(chunk):
    matrix = np.zeros((len(chunk), 12+24))
    for i,accord in enumerate(chunk):
        partition = accord.split("_")
        note = partition[0]
        if dico_notes[note] != -1:
            matrix[i][dico_notes[note]] = 1
        accord = "_".join(partition[1:])
        for j in range(12,12+24):
            matrix[i][j] = dico_res[accord][j-12]
    return matrix


In [15]:
def get_chunks(filename):
    data = pd.read_csv(filename, sep=",", header=None)
    chunks = {}
    chunk = []
    title = None
    for i, row in data.iterrows():
        if pd.notna(row[0]):
            if title is not None:
                chunks[title] = chunk
            title = row[0]
            chunk = []
        else:
            chunk.append(row[2])
    
    if title is not None:
        chunks[title] = chunk
    return chunks

In [19]:
get_chunks("cross-era_chords-chordino\chords-chordino_orchestra_baroque.csv")

<>:1: SyntaxWarning:

invalid escape sequence '\c'

<>:1: SyntaxWarning:

invalid escape sequence '\c'

C:\Users\PCAJM\AppData\Local\Temp\ipykernel_16144\300050348.py:1: SyntaxWarning:

invalid escape sequence '\c'



{'orchestra_baroque/CrossEra-0001_Albinoni__sinata_a_cinque_no._6_in_g_minor_op._2_1_adagio.mp3': ['G_min',
  'D_min',
  'Eb_maj',
  'Bb_maj',
  'A_dim',
  'D_maj',
  'A_min_min7',
  'G_maj',
  'C_min',
  'D_maj_min7',
  'C_min',
  'D_maj_min7',
  'G_min',
  'D_maj',
  'G_min',
  'A_dim_min7',
  'F_maj',
  'Bb_maj',
  'Eb_maj',
  'Bb_maj',
  'Eb_maj_maj7',
  'F_maj',
  'Bb_min',
  'F_min',
  'C_min',
  'D_maj_min7',
  'C_min_min7',
  'D_maj_min7',
  'G_min',
  'A_min_min7',
  'G_min',
  'B_dim',
  'C_min',
  'B_dim_min7',
  'C_min',
  'D_maj_min7',
  'G_min',
  'D_min',
  'Eb_maj',
  'Bb_maj',
  'Ab_maj_maj7',
  'D_maj',
  'G_min',
  'N',
  'N'],
 'orchestra_baroque/CrossEra-0002_Albinoni__sinata_a_cinque_no._6_in_g_minor_op._2_2_allegro.mp3': ['G_maj',
  'D_maj',
  'G_min_min7',
  'C_min',
  'G_min',
  'D_min',
  'A_maj',
  'D_min',
  'C_maj',
  'F_maj',
  'G_min',
  'D_min',
  'G_min',
  'F_maj',
  'Bb_maj',
  'C_min_min7',
  'G_min',
  'D_min',
  'C_maj',
  'F_maj',
  'G_min_min7',
